# Next-trade reaction: a 2,000-trade walkthrough

This is a small V8 research example, not a replacement for V7 or a trading strategy.
We use AAPL on 2026-08-20, taking the first 2,000 eligible canonical trades after 09:30 ET.
There are no candles, visit IDs, price bins, learned thresholds, or deep models.

1. Load time, price, and trade size.
2. Calculate the six backward/forward differences.
3. Inspect the nine arrival/departure categories using **initial history only**.
4. Predict from event **1,001** onward, updating only when an outcome becomes observable.
5. Evaluate 999 completed predictions; event 2,000 remains pending.

All source and output artifacts are saved outside the repository. Run this notebook with the `ml4t` Python environment. Save an executed copy in the runtime directory; keep the source notebook output-free.

In [ ]:
import os
import sys
from pathlib import Path
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True

REPO = Path(r"D:/TradingCodes/quant-research-workbench")
RUNTIME_ROOT = Path(r"D:/TradingML/runtimes")
assert REPO.is_dir() and RUNTIME_ROOT.is_dir(), "Required source/runtime root is unavailable"
sys.path.insert(0, str(REPO))

import json
from datetime import datetime, timezone
import hashlib
import subprocess
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from IPython.display import display
from research.level_book.v8.reaction_demo_source import load_sample

TICKER = "AAPL"
DAY = "2026-08-20"
N_EVENTS = 2000
INITIAL_EVENTS = 1000
PRIOR_STRENGTH = 10.0  # Fixed before evaluation; not selected on the second half.
RUN = RUNTIME_ROOT / "level-book-v8" / "reaction-demo" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN.mkdir(parents=True, exist_ok=False)
print("Outputs:", RUN)
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(12)

## 1. Load the trades
The helper does only source validation and a bounded read. It reads the certified canonical SIP table, verifies reporting coverage, excludes delayed and unknown-clock trades, and requires price/volume-eligible conditions. No flatfiles or provider requests are used.

We order by SIP timestamp and canonical ordinal. Equal timestamps retain their deterministic sequence; their time difference is zero. Thus the experiment models the ordered canonical feed, not a claim about real broker execution order. "Next trade" means the next **eligible** trade.

Price uses exact integer units (1/10,000 dollar) for grouping. Volume means individual trade size, not cumulative volume. Ordinal is provenance/tie-breaking metadata, not a model input.

In [ ]:
rows, provenance = load_sample(TICKER, DAY, N_EVENTS)
raw = pl.DataFrame(rows).with_row_index("event", offset=1).with_columns(
    pl.from_epoch("time_us", time_unit="us").dt.replace_time_zone("UTC").alias("time"),
    (pl.col("price_units") / 10000).alias("price"),
)
assert raw.height == N_EVENTS
assert raw["ordinal"].n_unique() == N_EVENTS
assert raw["time_us"].is_sorted()
raw.write_parquet(RUN / "source_trades.parquet")
(RUN / "source_provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
display(raw.select("event", "time", "price", "volume").head(12))
display(pl.DataFrame(provenance["audit"]))
print("Loaded time span:", raw["time"][0], "to", raw["time"][-1])
print("Unique prices:", raw["price_units"].n_unique())

## 2. Step one: six vectorized differences
For each original column x, backward = current − previous and forward = next − current.
Time differences below are seconds. Price differences are dollars. Volume differences are changes in trade size.

The full table below is for explanation and later scoring. **Forward columns are labels/outcomes, never prediction inputs.** At any earlier cutoff, they must be recomputed on that prefix. Null boundaries remain null.

In [ ]:
def step_one(frame):
    return frame.with_columns(
        ((pl.col("time_us") - pl.col("time_us").shift(1)) / 1e6).alias("time_backward"),
        ((pl.col("time_us").shift(-1) - pl.col("time_us")) / 1e6).alias("time_forward"),
        ((pl.col("price_units") - pl.col("price_units").shift(1)) / 10000).alias("price_backward"),
        ((pl.col("price_units").shift(-1) - pl.col("price_units")) / 10000).alias("price_forward"),
        (pl.col("volume") - pl.col("volume").shift(1)).alias("volume_backward"),
        (pl.col("volume").shift(-1) - pl.col("volume")).alias("volume_forward"),
    ).with_columns(
        pl.col("price_backward").sign().cast(pl.Int8).alias("arrival"),
        pl.col("price_forward").sign().cast(pl.Int8).alias("departure"),
    )

step1 = step_one(raw)
columns = ["event", "time", "price", "volume", "time_backward", "time_forward",
           "price_backward", "price_forward", "volume_backward", "volume_forward"]
display(step1.select(columns).head(15))
display(step1.select(columns).tail(3))
assert step1["price_backward"][0] is None
assert step1["price_forward"][-1] is None

## 3. Aggregate the initial 1,000 events only
We recompute shifts after slicing. At event 1,000 its own departure is still unknown.
Group by exact price, arrival direction and departure direction; no visits are introduced.

Arrival/departure codes: −1 down, 0 unchanged, +1 up. The nine categories are:

| Arrival / departure | Down | Unchanged | Up |
|---|---|---|---|
| Down | Down continuation | Pause after decline | Support-like bounce |
| Unchanged | Down after unchanged | Unchanged / unchanged | Up after unchanged |
| Up | Resistance-like rejection | Pause after rise | Up continuation |

These labels describe adjacent trades, not sustained structural outcomes. The first row has unknown arrival and the final prefix row has unknown departure; both remain visible in the audit rather than being classified as unchanged.

In [ ]:
initial = step_one(raw.head(INITIAL_EVENTS))
complete = initial.filter(pl.col("arrival").is_not_null() & pl.col("departure").is_not_null())
profile = complete.group_by("price_units", "price", "arrival", "departure").agg(
    pl.len().alias("trade_count"),
    pl.col("volume").sum().alias("traded_volume"),
    pl.col("time_forward").sum().alias("hold_seconds"),
    pl.col("price_backward").mean().alias("mean_arrival_move"),
    pl.col("price_forward").mean().alias("mean_departure_move"),
    pl.col("volume_backward").mean().alias("mean_volume_backward"),
    pl.col("volume_forward").mean().alias("mean_volume_forward"),
).sort("price_units", "arrival", "departure")
print("Complete classified rows:", complete.height, "; boundary rows:", initial.height - complete.height)
display(profile.head(18))
# Show all nine cells for the busiest price in initial history only.
busiest = complete.group_by("price_units").len().sort(["len", "price_units"], descending=[True, False])["price_units"][0]
grid = pl.DataFrame({"arrival": [-1, -1, -1, 0, 0, 0, 1, 1, 1], "departure": [-1, 0, 1] * 3}).cast(pl.Int8)
display(grid.join(profile.filter(pl.col("price_units") == busiest), on=["arrival", "departure"], how="left")
        .select("arrival", "departure", pl.col("trade_count", "traded_volume", "hold_seconds").fill_null(0)))
print("Price shown:", busiest / 10000)

## 4. A deliberately small probability model
For the current exact price and arrival direction, count previously resolved down / unchanged / up outcomes.

To avoid 100% confidence from one observation, shrink those counts toward the historical unconditional direction distribution:

`global_probability = (global_counts + 1) / (global_count + 3)`

`prediction = (local_counts + 10 * global_probability) / (local_count + 10)`

An unseen price/arrival pair uses the global probabilities exactly. Ten is a fixed illustrative prior strength, not a fitted hyperparameter. The baseline is that same global distribution, without conditioning on price or arrival.

Time and volume differences are shown and aggregated, but deliberately are **not model inputs yet**. This lets us evaluate the simplest model discussed before introducing more complexity.

The loop below represents live arrival, not vectorized retrospective fitting. When event j arrives, it reveals event j−1's outcome. We update that outcome, then predict event j's next direction. Prediction starts at event 1,001, after the first 1,000 events. Previously evaluated outcomes may train later predictions: this is an expanding-history, walk-forward evaluation.

In [ ]:
def walk_forward(prices, initial_events=1000, prior_strength=10.0):
    prices = np.asarray(prices, dtype=np.int64)
    global_counts = np.zeros(3, dtype=np.int64)  # down, unchanged, up
    local_counts = {}
    forecasts = []
    for i in range(len(prices)):
        if i >= 1:
            # This outcome only becomes known NOW, when event i arrives.
            resolved_class = int(np.sign(prices[i] - prices[i - 1])) + 1
            global_counts[resolved_class] += 1
            if i >= 2:
                previous_arrival = int(np.sign(prices[i - 1] - prices[i - 2]))
                previous_key = (int(prices[i - 1]), previous_arrival)
                local_counts.setdefault(previous_key, np.zeros(3, dtype=np.int64))[resolved_class] += 1
        if i < initial_events:
            continue
        arrival = int(np.sign(prices[i] - prices[i - 1]))
        counts = local_counts.get((int(prices[i]), arrival), np.zeros(3, dtype=np.int64))
        baseline = (global_counts + 1) / (global_counts.sum() + 3)
        probability = (counts + prior_strength * baseline) / (counts.sum() + prior_strength)
        forecasts.append(dict(
            event=i + 1, arrival=arrival, matching_history=int(counts.sum()),
            known_outcomes=int(global_counts.sum()),
            p_down=float(probability[0]), p_unchanged=float(probability[1]), p_up=float(probability[2]),
            baseline_down=float(baseline[0]), baseline_unchanged=float(baseline[1]), baseline_up=float(baseline[2]),
        ))
    return pl.DataFrame(forecasts)

predictions = walk_forward(raw["price_units"].to_numpy(), INITIAL_EVENTS, PRIOR_STRENGTH)
# Actual outcomes are attached AFTER every forecast has been generated.
predictions = predictions.join(
    step1.select("event", "time", "price", "volume", "time_backward", "price_backward", "volume_backward",
                 pl.col("departure").alias("actual")), on="event", how="left")
display(predictions.select("event", "price", "arrival", "matching_history", "p_down", "p_unchanged", "p_up", "actual").head(12))
display(predictions.tail(1))
assert predictions.height == 1000
assert predictions["known_outcomes"][0] == 1000
assert predictions["actual"][-1] is None

## 5. Evaluate the 999 resolved forecasts
Accuracy uses the largest probability (ties use down, then unchanged, then up). Balanced accuracy averages recall over classes present in this sample. Log loss evaluates probability quality; lower is better. Multiclass Brier score is the mean sum of squared probability errors; lower is better (range 0–2).

We compare against an unconditional model updated on exactly the same available outcomes. The final unresolved prediction is retained but excluded from scoring. This small, single-ticker sample cannot establish trading profitability or generalization.

In [ ]:
scored = predictions.filter(pl.col("actual").is_not_null())
y = scored["actual"].to_numpy().astype(int) + 1
p = scored.select("p_down", "p_unchanged", "p_up").to_numpy()
b = scored.select("baseline_down", "baseline_unchanged", "baseline_up").to_numpy()

def evaluate(probabilities):
    chosen = probabilities.argmax(axis=1)
    confusion = np.zeros((3, 3), dtype=int)
    np.add.at(confusion, (y, chosen), 1)
    support = confusion.sum(axis=1)
    recalls = np.divide(confusion.diagonal(), support, out=np.full(3, np.nan), where=support > 0)
    return dict(accuracy=float(np.mean(chosen == y)), balanced_accuracy=float(np.nanmean(recalls)),
                log_loss=float(-np.log(probabilities[np.arange(len(y)), y]).mean()),
                brier=float(np.square(probabilities - np.eye(3)[y]).sum(axis=1).mean())), confusion

model_metrics, confusion = evaluate(p)
baseline_metrics, _ = evaluate(b)
metrics = pl.DataFrame([dict(model="price + arrival", **model_metrics), dict(model="unconditional baseline", **baseline_metrics)])
display(metrics)
display(pl.DataFrame(confusion, schema=["predicted_down", "predicted_unchanged", "predicted_up"], orient="row")
        .with_columns(pl.Series("actual", ["down", "unchanged", "up"])))
print("Scored:", len(y), "; pending:", predictions.height - len(y))
print("Actual class counts (down, unchanged, up):", np.bincount(y, minlength=3))
print("Predictions with zero matching history:", int((scored["matching_history"] == 0).sum()))
print("Accuracy difference:", model_metrics["accuracy"] - baseline_metrics["accuracy"])
assert len(y) == 999
assert np.allclose(p.sum(axis=1), 1) and np.all(p > 0)

## 6. Visualize and verify chronology
The checks compare predictions against a shorter prefix and against changed future prices. Earlier predictions must be identical. A hand-calculated toy sequence also verifies the first conditional prediction.

In [ ]:
prices = raw["price_units"].to_numpy()
prob_cols = ["p_down", "p_unchanged", "p_up", "matching_history", "known_outcomes"]
prefix = walk_forward(prices[:1100], INITIAL_EVENTS, PRIOR_STRENGTH)
assert prefix.select(prob_cols).equals(predictions.head(100).select(prob_cols))
changed = prices.copy()
changed[1100:] += 100000
assert walk_forward(changed, INITIAL_EVENTS, PRIOR_STRENGTH).head(100).select(prob_cols).equals(prefix.select(prob_cols))
toy = walk_forward([100, 101, 100, 101, 100], initial_events=3, prior_strength=10)
# At index 3: global counts=[1,0,2], local (101,+1) counts=[1,0,0].
expected = (np.array([1,0,0]) + 10 * np.array([2,1,3]) / 6) / 11
assert np.allclose(toy.select("p_down", "p_unchanged", "p_up").row(0), expected)
print("PASS: prefix invariance, future perturbation, hand-calculated forecast, probability sums and boundary counts")

fig, axes = plt.subplots(3, 1, figsize=(11, 9), constrained_layout=True)
axes[0].plot(raw["event"], raw["price"], linewidth=0.8)
axes[0].axvline(INITIAL_EVENTS + 0.5, color="black", linestyle="--", label="prediction starts")
axes[0].set(ylabel="Trade price", title=f"{TICKER} {DAY}: 2,000 eligible trades")
axes[0].legend()
x = scored["event"].to_numpy()
for name, probabilities in [("price + arrival", p), ("unconditional baseline", b)]:
    hit = probabilities.argmax(axis=1) == y
    loss = -np.log(probabilities[np.arange(len(y)), y])
    axes[1].plot(x, np.cumsum(hit) / np.arange(1, len(y)+1), label=name)
    axes[2].plot(x, np.cumsum(loss) / np.arange(1, len(y)+1), label=name)
axes[1].set(ylabel="Cumulative accuracy")
axes[2].set(ylabel="Cumulative log loss", xlabel="Event number")
axes[1].legend(); axes[2].legend()
fig.savefig(RUN / "evaluation.png", dpi=150)
plt.show()
plt.close(fig)

## 7. Save the result
Source snapshot, provenance, step-one table, initial profile, all predictions, metrics, and chart remain under the run directory. The source notebook contains no generated outputs. The last forecast is pending; no 2,001st trade is fetched just to score it.

In [ ]:
step1.write_parquet(RUN / "step_one.parquet")
profile.write_parquet(RUN / "initial_profile.parquet")
predictions.write_parquet(RUN / "predictions.parquet")
notebook_path = REPO / "research/level_book/v8/next_trade_reaction.ipynb"
manifest = dict(ticker=TICKER, day=DAY, events=N_EVENTS, initial_events=INITIAL_EVENTS,
                scored=len(y), pending=1, prior_strength=PRIOR_STRENGTH,
                model=model_metrics, baseline=baseline_metrics,
                source_sha256=provenance["sample_sha256"],
                git_commit=subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip(),
                source_files_sha256={str(path.relative_to(REPO)): hashlib.sha256(path.read_bytes()).hexdigest()
                    for path in [notebook_path, REPO / "research/level_book/v8/reaction_demo_source.py"]},
                python=sys.version, numpy=np.__version__, polars=pl.__version__,
                checks="passed: prefix, future perturbation, hand example, probability and boundary invariants")
(RUN / "metrics.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved:", RUN)
display(metrics)